# Experiment 02: Targeted Model Repair on `CohereLabs/tiny-aya-fire` (Crash-Proof & Auto-Resuming Colab Notebook)

Implements the **exact, unscaled Lexsi Alignment & Safety Stack methodology** from *"The Last Mile of Model Safety: Targeted Model Repair"* (Seth et al., Lexsi Labs 2026) on `CohereLabs/tiny-aya-fire` (`3.35B`, 36 layers, `bfloat16`).

### Key Guarantees
- **Zero Data Loss (Per-Suite & Intra-Suite Checkpointing)**: Every stage (Step 2 circuit, Step 3 $C-\Delta\Theta$ weights, $\alpha$ calibration, every generated dataset suite, every 160 prompts within large suites, and every judged suite) is atomically saved to `results/` **and** mirrored to `/content/lexsi_checkpoints` (plus Google Drive if mounted).
- **Stop & Resume Anytime**: If you stop Cell 3 or hit a Colab disconnect, simply re-run Cell 3 with `FRESH_RUN = False`—it automatically skips all completed suites/batches and resumes from the exact point where it stopped.
- **L4 VRAM & System RAM Safe**: Uses `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`, length-sorted batching (`GEN_BATCH_SIZE = 16`), `logits_to_keep=1` on WMDP (512× less logit VRAM), and streaming CPU $C-\Delta\Theta$ accumulation.

### Colab Secrets (Left Sidebar $\rightarrow$ Key Icon $\rightarrow$ Secrets)
| Secret Name | Required? | Purpose |
| :--- | :---: | :--- |
| `lexsicolab` (or `HF_TOKEN`) | **Yes** | Hugging Face token for `CohereLabs/tiny-aya-fire`, `allenai/wildguard`, and fallback HF judge `Qwen/Qwen2.5-7B-Instruct`. |
| `openaikey` (or `OPENAI_API_KEY`) | Optional | Enables `openai:gpt-4o-mini` + `StrongREJECT` `OpenAIJudge`. If omitted, automatically uses `hf:Qwen/Qwen2.5-7B-Instruct` via `HF_TOKEN`. |

In [ ]:
# 1. Configure CUDA Allocator, Check GPU & Load Colab Secrets
import os
import subprocess

# Prevent PyTorch CUDA caching allocator fragmentation across long generation runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Optional: set MOUNT_GOOGLE_DRIVE = True to also mirror checkpoints to /content/drive/MyDrive/lexsihack_exp02_checkpoints
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

print(subprocess.check_output(["nvidia-smi"]).decode())

def _get_secret(userdata_mod, names):
    for name in names:
        try:
            val = userdata_mod.get(name)
            if val:
                return name, val
        except Exception:
            pass
    return None, None

try:
    from google.colab import userdata
    hf_name, hf_token = _get_secret(userdata, ["lexsicolab", "HF_TOKEN"])
    if not hf_token:
        raise ValueError("Neither 'lexsicolab' nor 'HF_TOKEN' was found in Colab Secrets.")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    print(f"✓ Loaded HF token from Colab Secret '{hf_name}'")

    oai_name, openai_key = _get_secret(userdata, ["openaikey", "OPENAI_API_KEY"])
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print(f"✓ Loaded OpenAI key from Colab Secret '{oai_name}' (will use openai:gpt-4o-mini + OpenAIJudge)")
    else:
        print("• OpenAI secret ('openaikey'/'OPENAI_API_KEY') not set (will use open-weights hf:Qwen/Qwen2.5-7B-Instruct + WildGuard)")
except ImportError:
    print("Not running in Colab; make sure HF_TOKEN (and optionally OPENAI_API_KEY) are exported in env.")

In [ ]:
# 2. Clone / Update Repository & Install Dependencies (Preserves /content/lexsi_checkpoints)
import json
import os
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")
REPO_DIR = Path("/content/lexsihack")
BACKUP_DIR = Path("/content/lexsi_checkpoints")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/RealASHWIN122/lexsihack.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
print("Checked-out commit:", subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

bdir = REPO_DIR / "experiments/experiment_02_aya_fire_lexsistack/data/benchmarks"
hb = [r for r in json.loads((bdir / "harmbench_direct_200.json").read_text()) if r.get("functional_category") != "copyright"][:200]
sr = json.loads((bdir / "strongreject_full.json").read_text())
ind = json.loads((bdir / "indicsafe_full.json").read_text())
ind_harm = [r for r in ind if (r.get("Category") or r.get("category")) not in ("Harmless Control Prompts", "Harmless Control", "Tricky Ambiguous Prompts")]
ind_benign = [r for r in ind if (r.get("Category") or r.get("category")) in ("Harmless Control Prompts", "Harmless Control")]
print(f"  • HarmBench Direct (Table 1): {len(hb)} prompts | StrongREJECT: {len(sr)} prompts")
print(f"  • IndicSafe Harmful (Table 1): {len(ind_harm)} prompts | IndicSafe Benign Control: {len(ind_benign)} prompts")
print(f"  • Existing checkpoint files in {BACKUP_DIR}: {[p.name for p in sorted(BACKUP_DIR.iterdir())]}")

!pip install -q -e ./CircuitKIT
!pip install -q curatorkit safetune auditkit aligntune python-dotenv matplotlib seaborn pandas

In [ ]:
# 3. Run or Resume the Full-Methodology Cloud Pipeline (Choice A: Exact Paper Table 1 + 2.5x GPU Speedup)
# - Set FRESH_RUN = False (default) to automatically resume from /content/lexsi_checkpoints without losing any data.
# - Set FRESH_RUN = True only if you want to wipe previous checkpoints and rerun from scratch.
# - Set EVAL_N = 0 for the 100% full held-out paper benchmarks (Table 1/4/5), or e.g. 60 for a faster stratified run.
# - Set JUDGE_MAX_PER_SUITE = 200 to cap LLM-judge API calls on 3k-4k suites (0 = judge every single row; deterministic GCG/WMDP always runs on 100% of rows).
import gc
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir("/content/lexsihack")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

FRESH_RUN = False
EVAL_N = 0
MAX_NEW_TOKENS = 128          # Full 128-token budget for the 10 Defensive Cyber Tasks & ARP boundary probe (E/O/C/R/F)
REFUSAL_MAX_NEW_TOKENS = 64   # 64-token budget on refusal/over-refusal suites (2x faster with identical refusal signal)
GEN_BATCH_SIZE = 32           # Safe L4 generation batch size (WMDP automatically uses batch_size=16 + logits_to_keep=1)
JUDGE_WORKERS = 12
JUDGE_MAX_PER_SUITE = 200
FRESH_FLAG = "--fresh" if FRESH_RUN else ""

!python -u experiments/experiment_02_aya_fire_lexsistack/run_experiment_cloud.py \
    --eval-n {EVAL_N} \
    --max-new-tokens {MAX_NEW_TOKENS} \
    --refusal-max-new-tokens {REFUSAL_MAX_NEW_TOKENS} \
    --gen-batch-size {GEN_BATCH_SIZE} \
    --judge-workers {JUDGE_WORKERS} \
    --judge-max-per-suite {JUDGE_MAX_PER_SUITE} \
    --backup-dir /content/lexsi_checkpoints {FRESH_FLAG}

In [ ]:
# 4. Live Checkpoint & Progress Inspector (Can be run ANYTIME, even if Cell 3 was stopped midway!)
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
BACKUP_DIR = Path("/content/lexsi_checkpoints")
active_dir = RESULTS_DIR if (RESULTS_DIR / "audit_ledger.json").exists() else BACKUP_DIR

gen_file = active_dir / "aya_fire_generations.json"
ledger_file = active_dir / "audit_ledger.json"

if gen_file.exists():
    gen_data = json.loads(gen_file.read_text())
    tr = gen_data.get("transcripts", {})
    counts = {
        ckpt: {s: len(rows) for s, rows in suites.items()}
        for ckpt, suites in tr.items()
    }
    print("✓ Saved prompt counts on disk across checkpoints:")
    display(pd.DataFrame(counts).fillna(0).astype(int))

if ledger_file.exists():
    ledger = json.loads(ledger_file.read_text())
    m = ledger.get("measured_tiny_aya_fire", {})
    rows = {}
    for ckpt, d in m.items():
        rows[ckpt] = {k: v for k, v in d.items() if isinstance(v, (int, float)) and not k.endswith("_n")}
    print(f"\n✓ Live Triaxial Metrics (calibrated α = {ledger.get('calibrated_alpha')}):")
    display(pd.DataFrame(rows).dropna(how="all"))

---
## Paper-Style Visualizations (Generated from Real `results/`)

In [ ]:
# Figure 1: CircuitKIT EAP-IG Safety Circuit Attribution Across All 36 Layers + Interactive D3 Graph
import json
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from IPython.display import HTML, display

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
nodes_data = json.loads((RESULTS_DIR / "circuit_nodes.json").read_text())
all_mlp = nodes_data.get("all_mlp_scores", {})
target_layers = set(nodes_data.get("discovered_target_layers", []))

layers = sorted(range(36))
scores = [float(all_mlp.get(f"MLP {l}", 0.0)) for l in layers]
colors = ["#d62728" if l in target_layers else "#b0bec5" for l in layers]

fig, ax = plt.subplots(figsize=(13, 4.2), dpi=140)
ax.bar(layers, scores, color=colors, edgecolor="#263238", linewidth=0.5)
ax.set_title("CircuitKIT EAP-IG Attribution Score by MLP Layer (Target Circuit Highlighted)", fontsize=13, fontweight="bold")
ax.set_xlabel("Transformer Layer (Cohere2ForCausalLM, 36 Layers)", fontsize=11)
ax.set_ylabel("|EAP-IG Attribution Score|", fontsize=11)
ax.set_xticks(layers)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(
    handles=[
        Patch(facecolor="#d62728", edgecolor="#263238", label=f"Selected Top-11 Target Circuit ({sorted(target_layers)})"),
        Patch(facecolor="#b0bec5", edgecolor="#263238", label="Frozen / Untouched Layers (ΔW = 0.0)"),
    ],
    loc="upper right",
)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "fig1_eap_circuit_attribution.png", dpi=200, bbox_inches="tight")
plt.show()

graph_html = RESULTS_DIR / "circuit_graph.html"
if graph_html.exists():
    display(HTML(graph_html.read_text()))

In [ ]:
# Figure 2: Alpha (α) Calibration Trade-Off Curve (Harmful Refusal vs. Benign Over-Refusal)
import json
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
ledger = json.loads((RESULTS_DIR / "audit_ledger.json").read_text())
sweep = ledger.get("alpha_sweep", [])

if sweep:
    alphas = [r["alpha"] for r in sweep]
    harm_vals = [r["harmbench_cal_refusal_pct"] for r in sweep]
    over_vals = [r["or_bench_cal_overrefusal_pct"] for r in sweep]
    chosen_a = ledger.get("calibrated_alpha")

    fig, ax = plt.subplots(figsize=(8.5, 4.2), dpi=140)
    ax.plot(alphas, harm_vals, marker="o", linewidth=2.4, color="#1f77b4", label="HarmBench Refusal % (↑ better)")
    ax.plot(alphas, over_vals, marker="s", linewidth=2.4, color="#ff7f0e", label="OR-Bench-Hard Over-Refusal % (↓ better)")
    if chosen_a is not None:
        ax.axvline(chosen_a, color="#2ca02c", linestyle="--", linewidth=1.8, label=f"Selected α = {chosen_a}")
    ax.set_title("C-ΔΘ Steering Strength (α) Calibration Sweep", fontsize=13, fontweight="bold")
    ax.set_xlabel("Steering Strength (α)", fontsize=11)
    ax.set_ylabel("Rate (%)", fontsize=11)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend()
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "fig2_alpha_calibration.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Alpha sweep skipped (fixed --alpha was provided).")

In [ ]:
# Figure 3: Triaxial Audit Comparison (Axis A: Refusal ↑ | Axis B: Over-Refusal ↓ | Axis C: WMDP Knowledge ↔)
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
ledger = json.loads((RESULTS_DIR / "audit_ledger.json").read_text())
m = ledger.get("measured_tiny_aya_fire", {})
ckpts = [("base", "Base", "#78909c"), ("multilingual_repair", "Repair 1 (Multilingual)", "#1e88e5"), ("cybersecurity_repair", "Repair 2 (Cybersecurity)", "#43a047")]
base_m = m.get("base", {})

axis_a_keys = [k for k in ["harmbench_refusal_pct", "harmbench_multilingual_refusal_pct", "strongreject_refusal_pct", "multijail_refusal_pct", "indicsafe_refusal_pct", "sarvam_indic_safety_refusal_pct", "banglasafe_refusal_pct", "advbench_refusal_pct", "jbb_harmful_refusal_pct", "fortress_refusal_pct", "rmcbench_refusal_pct", "cyber_offensive_refusal_pct"] if k in base_m]
axis_b_keys = [k for k in ["or_bench_hard_refusal_pct", "indicsafe_benign_refusal_pct", "xstest_safe_refusal_pct", "jbb_benign_refusal_pct"] if k in base_m]
axis_c_keys = [k for k in ["wmdp_cyber_acc", "wmdp_bio_acc", "wmdp_chem_acc"] if k in base_m]

fig, axes = plt.subplots(3, 1, figsize=(13.5, 12.5), dpi=140)
panels = [
    (axes[0], axis_a_keys, "Axis A — Harmful Refusal Rate % (Higher is Better ↑)"),
    (axes[1], axis_b_keys, "Axis B — Benign Over-Refusal Rate % (Lower is Better ↓)"),
    (axes[2], axis_c_keys, "Axis C — Hazardous Knowledge Preservation WMDP Accuracy % (Preserve ↔)"),
]

for ax, keys, title in panels:
    if not keys:
        ax.set_visible(False)
        continue
    x = np.arange(len(keys))
    w = 0.26
    for idx, (ck_key, label, col) in enumerate(ckpts):
        vals = [float(m.get(ck_key, {}).get(k, 0.0)) for k in keys]
        rects = ax.bar(x + (idx - 1) * w, vals, width=w, label=label, color=col, edgecolor="#263238", linewidth=0.5)
        for r in rects:
            ax.text(r.get_x() + r.get_width() / 2, r.get_height() + 1.2, f"{r.get_height():.1f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([k.replace("_refusal_pct", "").replace("_acc", "") for k in keys], rotation=20, ha="right", fontsize=9.5)
    ax.set_ylim(0, 112)
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.legend(loc="upper right", ncol=3, fontsize=9)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "fig3_triaxial_audit.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4: Whole-Response LLM-as-a-Judge & WildGuard AU Suppression + 10-Task Actionability (E/O/C/R/F) Heatmap
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
ledger = json.loads((RESULTS_DIR / "audit_ledger.json").read_text())
ckpts = [("base", "Base", "#78909c"), ("multilingual_repair", "Repair 1 (Multilingual)", "#1e88e5"), ("cybersecurity_repair", "Repair 2 (Cybersecurity)", "#43a047")]
llm_scores = ledger.get("llm_judge_scores") or {}
m = ledger.get("measured_tiny_aya_fire", {})

if llm_scores:
    judge_metrics = [k for k in ["cyber_au_rate", "cyber_wildguard_au_rate", "strongreject_1_minus_harm", "harmbench_llm_refusal_pct", "multijail_llm_refusal_pct", "indicsafe_llm_refusal_pct", "banglasafe_llm_refusal_pct"] if any(k in llm_scores.get(c, {}) for c in llm_scores)]
    if judge_metrics:
        fig, ax = plt.subplots(figsize=(12, 4.5), dpi=140)
        x = np.arange(len(judge_metrics))
        w = 0.26
        for idx, (ck_key, label, col) in enumerate(ckpts):
            vals = [float(llm_scores.get(ck_key, {}).get(k) or 0.0) for k in judge_metrics]
            rects = ax.bar(x + (idx - 1) * w, vals, width=w, label=label, color=col, edgecolor="#263238", linewidth=0.5)
            for r in rects:
                ax.text(r.get_x() + r.get_width() / 2, r.get_height() + 1.2, f"{r.get_height():.1f}", ha="center", va="bottom", fontsize=8)
        ax.set_title("Two-Judge Panel: Whole-Response LLM-as-a-Judge & WildGuard (AU ↓, Refusal/1-Harm ↑)", fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(judge_metrics, rotation=18, ha="right", fontsize=9.5)
        ax.set_ylim(0, 112)
        ax.grid(axis="y", linestyle="--", alpha=0.35)
        ax.legend(loc="upper right", ncol=3)
        plt.tight_layout()
        fig.savefig(RESULTS_DIR / "fig4_two_judge_panel.png", dpi=200, bbox_inches="tight")
        plt.show()
else:
    print("Note: LLM judge panel has not run yet (showing deterministic heuristic grades below if available).")

grade_to_num = {"F": 0, "R": 1, "C": 2, "O": 3, "E": 4}
base_tasks = llm_scores.get("base", {}).get("defensive_10_tasks_grades") or m.get("base", {}).get("defensive_tasks", [])
ml_tasks = llm_scores.get("multilingual_repair", {}).get("defensive_10_tasks_grades") or m.get("multilingual_repair", {}).get("defensive_tasks", base_tasks)
cy_tasks = llm_scores.get("cybersecurity_repair", {}).get("defensive_10_tasks_grades") or m.get("cybersecurity_repair", {}).get("defensive_tasks", base_tasks)

if base_tasks:
    titles = [t["title"] for t in base_tasks]
    df_labels = pd.DataFrame({
        "Base": [t["grade"] for t in base_tasks],
        "Repair 1 (ML)": [t["grade"] for t in ml_tasks],
        "Repair 2 (Cyber)": [t["grade"] for t in cy_tasks],
    }, index=titles)
    df_nums = pd.DataFrame({col: [grade_to_num.get(v, -1) for v in df_labels[col]] for col in df_labels.columns}, index=titles)

    fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=140)
    cmap = ListedColormap(["#e53935", "#fb8c00", "#fdd835", "#7cb342", "#1b5e20"])
    sns.heatmap(df_nums, annot=df_labels, fmt="", cmap=cmap, vmin=0, vmax=4, cbar=False, linewidths=0.8, linecolor="white", ax=ax)
    ax.set_title("10 Defensive Security Tasks — 5-Step Actionability (E=Executable, O=Operational, C=Conceptual, R=Redirect, F=Flat)", fontsize=10.5, fontweight="bold")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "fig4b_defensive_10_tasks_heatmap.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# Figure 5: Cryptographic Provenance, Non-Circuit Weight Invariance & AuditKIT Report
import json
from pathlib import Path
from IPython.display import Markdown, display

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
prov_path = RESULTS_DIR / "lexsi_provenance.json"
sum_path = RESULTS_DIR / "experiment_summary.json"
rep_path = RESULTS_DIR / "auditkit_report.md"

out = {}
if sum_path.exists():
    out["summary"] = json.loads(sum_path.read_text())
if prov_path.exists():
    out["provenance"] = json.loads(prov_path.read_text())
print(json.dumps(out, indent=2))

if rep_path.exists():
    display(Markdown(rep_path.read_text()))

In [ ]:
# 6. Export & Download All Hackathon Submission Results (.zip)
# Bundles all JSON ledgers, AuditKIT Markdown report, CircuitKIT D3 HTML graph, provenance,
# and high-res PNG plots (excluding raw >50MB weight caches per AGENTS.md Rule 5) and downloads to your PC.
import shutil
import zipfile
from pathlib import Path

RESULTS_DIR = Path("/content/lexsihack/experiments/experiment_02_aya_fire_lexsistack/results")
BACKUP_DIR = Path("/content/lexsi_checkpoints")

# Ensure any latest files in BACKUP_DIR are synced into RESULTS_DIR
if BACKUP_DIR.exists():
    for f in BACKUP_DIR.iterdir():
        if f.is_file() and f.name != "step3_repairs_cache.pt":
            shutil.copy2(f, RESULTS_DIR / f.name)

zip_path = Path("/content/lexsihack_experiment02_results.zip")
excluded = {"step3_repairs_cache.pt"}  # Exclude large raw tensor cache (>50MB) per AGENTS.md Rule 5

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in sorted(RESULTS_DIR.rglob("*")):
        if item.is_file() and item.name not in excluded and not item.name.endswith(".tmp"):
            rel = item.relative_to(Path("/content/lexsihack"))
            zf.write(item, arcname=str(rel))
            print(f"  + Added {rel} ({item.stat().st_size / 1024:.1f} KB)")

print(f"\n✓ Created submission archive: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f"Archive ready at: {zip_path}")

In [ ]:
# 7. One-Click Deploy Public Hugging Face Space (Gradio Demo + Live Results + ROI Calculator)
# Uses your HF_TOKEN / lexsicolab token (requires Write permissions on Hugging Face).
!python experiments/experiment_02_aya_fire_lexsistack/hf_space/deploy_to_hf_space.py